In [ ]:
# ============================================================
# Hybrid LSTM-GARCH — Step 2: EDA and Statistical Tests
# ============================================================

import matplotlib.gridspec as gridspec
from scipy import stats
from statsmodels.stats.stattools import jarque_bera
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.stats.diagnostic import het_arch

In [ ]:
# ── Summary statistics ────────────────────────────────────────
print("=== SPY Log Return Summary Statistics ===")
print(f"Observations:       {len(returns):,}")
print(f"Mean daily return:  {returns.mean():.6f}  ({returns.mean()*252:.2%} annualised)")
print(f"Std deviation:      {returns.std():.6f}  ({returns.std()*np.sqrt(252):.2%} annualised vol)")
print(f"Skewness:           {returns.skew():.4f}")
print(f"Excess kurtosis:    {returns.kurt():.4f}")
print(f"Min (worst day):    {returns.min():.4f}  ({returns.idxmin().date()})")
print(f"Max (best day):     {returns.max():.4f}  ({returns.idxmax().date()})")

In [ ]:
# ── Test 1: Jarque-Bera normality test ────────────────────────
print("\n=== Test 1: Jarque-Bera Normality Test ===")
jb_stat, jb_p, jb_skew, jb_kurt = jarque_bera(returns)
print(f"JB Statistic:  {jb_stat:,.2f}")
print(f"P-value:       {jb_p:.6f}")
print(f"Skewness:      {jb_skew:.4f}  (negative = left tail heavier)")
print(f"Kurtosis:      {jb_kurt:.4f}  (>3 = fat tails vs normal)")
print(f"Conclusion:    {'Returns are NOT normally distributed — fat tails confirmed' if jb_p < 0.05 else 'Cannot reject normality'}")


In [ ]:
# ── Test 2: ADF stationarity test ─────────────────────────────
print("\n=== Test 2: Augmented Dickey-Fuller Stationarity Test ===")
adf_stat, adf_p, adf_lags, adf_n, adf_cv, _ = adfuller(returns, maxlag=20, autolag='AIC')
print(f"ADF Statistic: {adf_stat:.4f}")
print(f"P-value:       {adf_p:.6f}")
print(f"Lags used:     {adf_lags}")
print(f"Critical values:")
for k, v in adf_cv.items():
    print(f"   {k}: {v:.4f}")
print(f"Conclusion:    {'Series is STATIONARY — safe to model directly' if adf_p < 0.05 else 'Non-stationary — differencing needed'}")


In [ ]:
# ── Test 3: ARCH LM test (volatility clustering) ──────────────
print("\n=== Test 3: Engle ARCH LM Test (Volatility Clustering) ===")
lm_stat, lm_p, f_stat, f_p = het_arch(returns, nlags=10)
print(f"LM Statistic:  {lm_stat:.4f}")
print(f"P-value:       {lm_p:.6f}")
print(f"F-statistic:   {f_stat:.4f}")
print(f"F p-value:     {f_p:.6f}")
print(f"Conclusion:    {'ARCH effects CONFIRMED — GARCH modeling is justified' if lm_p < 0.05 else 'No significant ARCH effects detected'}")

In [ ]:
# ── Plot 02: Return distribution ──────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

In [ ]:
# Histogram vs Normal
ax1 = fig.add_subplot(gs[0, 0])
x = np.linspace(returns.min(), returns.max(), 300)
normal_pdf = stats.norm.pdf(x, returns.mean(), returns.std())
ax1.hist(returns, bins=120, density=True,
         color='#2166ac', alpha=0.6, label='Actual returns')
ax1.plot(x, normal_pdf, 'r-', linewidth=2, label='Normal distribution')
ax1.set_title('Return Distribution vs Normal\n(notice the fat tails)',
              fontweight='bold', fontsize=11)
ax1.set_xlabel('Log Return')
ax1.set_ylabel('Density')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

In [ ]:
# Q-Q plot
ax2 = fig.add_subplot(gs[0, 1])
(osm, osr), (slope, intercept, r) = stats.probplot(returns, dist='norm')
ax2.plot(osm, osr, '.', color='#2166ac', markersize=1.5, alpha=0.4)
ax2.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=1.5)
ax2.set_title('Q-Q Plot\n(S-curve = heavy tails, not normal)',
              fontweight='bold', fontsize=11)
ax2.set_xlabel('Theoretical Quantiles (Normal)')
ax2.set_ylabel('Sample Quantiles')
ax2.grid(alpha=0.3)

In [ ]:
# ACF of returns
ax3 = fig.add_subplot(gs[1, 0])
acf_vals = acf(returns, nlags=40, alpha=0.05)
lags = range(len(acf_vals[0]))
ci_lower = [acf_vals[1][i][0] - acf_vals[0][i] for i in lags]
ci_upper = [acf_vals[1][i][1] - acf_vals[0][i] for i in lags]
ax3.bar(lags, acf_vals[0], color='#2166ac', alpha=0.7, width=0.6)
ax3.fill_between(lags, ci_lower, ci_upper, alpha=0.2, color='gray')
ax3.axhline(0, color='black', linewidth=0.5)
ax3.set_title('ACF of Returns\n(little autocorrelation — returns hard to predict)',
              fontweight='bold', fontsize=11)
ax3.set_xlabel('Lag (days)')
ax3.set_ylabel('Autocorrelation')
ax3.grid(alpha=0.3)

In [ ]:
# ACF of squared returns
ax4 = fig.add_subplot(gs[1, 1])
sq_returns = returns ** 2
acf_sq = acf(sq_returns, nlags=40, alpha=0.05)
lags_sq = range(len(acf_sq[0]))
ci_lower_sq = [acf_sq[1][i][0] - acf_sq[0][i] for i in lags_sq]
ci_upper_sq = [acf_sq[1][i][1] - acf_sq[0][i] for i in lags_sq]
ax4.bar(lags_sq, acf_sq[0], color='#d6604d', alpha=0.7, width=0.6)
ax4.fill_between(lags_sq, ci_lower_sq, ci_upper_sq, alpha=0.2, color='gray')
ax4.axhline(0, color='black', linewidth=0.5)
ax4.set_title('ACF of Squared Returns\n(strong persistence = ARCH effects confirmed!)',
              fontweight='bold', fontsize=11, color='#d6604d')
ax4.set_xlabel('Lag (days)')
ax4.set_ylabel('Autocorrelation')
ax4.grid(alpha=0.3)

fig.suptitle('SPY Log Returns — Statistical Properties (1993–2024)',
             fontsize=14, fontweight='bold')
plt.savefig('outputs/02_return_statistics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/02_return_statistics.png")

In [ ]:
# ── Plot 03: Rolling volatility ───────────────────────────────
fig2, ax = plt.subplots(figsize=(14, 5))

rolling_vol = returns.rolling(window=21).std() * np.sqrt(252) * 100
ax.fill_between(spy.index, rolling_vol, alpha=0.35, color='#d6604d')
ax.plot(spy.index, rolling_vol, color='#d6604d', linewidth=0.6)
ax.axhline(rolling_vol.mean(), color='black', linestyle='--',
           linewidth=1, label=f'Mean vol: {rolling_vol.mean():.1f}%')

crises = [
    ('2000-03-01', '2002-10-01', 'Dot-com bust'),
    ('2007-10-01', '2009-06-01', '2008 Crisis'),
    ('2020-02-01', '2020-08-01', 'COVID'),
    ('2022-01-01', '2022-12-31', '2022 Rate hikes')
]
for start, end, label in crises:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.1, color='purple')
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ax.text(mid, rolling_vol.max() * 0.9, label,
            ha='center', fontsize=8.5, color='purple', fontweight='bold')

ax.set_title('SPY 21-Day Rolling Annualised Volatility (1993–2024)\n'
             'Volatility clustering clearly visible across all crisis periods',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Annualised Volatility (%)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/03_rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/03_rolling_volatility.png")